# Phase 1 — Data Pipeline & Evaluation Metrics
**Project:** Can LoRA-Adapted SAM Match Specialist Polyp Networks?

This notebook covers the full Phase 1 deliverables:
1. Install dependencies
2. Clone repo
3. Download all five polyp datasets
4. Verify PraNet train/seen/unseen splits
5. Visualise sample images from each dataset
6. Smoke-test all six evaluation metrics on real images

## 1. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

In [ ]:
if IN_COLAB:
    !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git
    %cd msu2026summer_final_project
    !pip install -q -r requirements.txt
else:
    import os
    # If running locally, cd to the repo root before running
    print('Running locally — make sure you are in the repo root.')

In [ ]:
import os, sys
# Ensure repo root is on the path
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

DATA_ROOT = Path('data/polyp')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Data root: {DATA_ROOT.resolve()}')

## 2. Download Datasets

We use the **standard PraNet data package** which contains all five datasets
pre-organised into `TrainDataset/` and `TestDataset/` with the exact split
used by PraNet, Polyp-PVT, and all subsequent polyp papers.

**Download source:** The PraNet GitHub README
([github.com/DengPingFan/PraNet](https://github.com/DengPingFan/PraNet))
links to a public Google Drive package containing all five datasets.
Copy the file ID from that link and paste it below.

Alternatively, Kvasir-SEG is available as a direct download from Simula Research Lab.

In [ ]:
# -------------------------------------------------------------------------
# Option A: Download Kvasir-SEG directly (no Google Drive needed)
# -------------------------------------------------------------------------
kvasir_dir = DATA_ROOT / 'TrainDataset' / 'Kvasir'

if not kvasir_dir.exists():
    print('Downloading Kvasir-SEG from Simula Research Lab...')
    !wget -q --show-progress \
        https://datasets.simula.no/kvasir-seg/Kvasir-SEG.zip \
        -O /tmp/Kvasir-SEG.zip
    !unzip -q /tmp/Kvasir-SEG.zip -d /tmp/kvasir_raw

    # Kvasir-SEG ships as images/ + masks/ — map to our expected layout
    import shutil
    kvasir_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree('/tmp/kvasir_raw/Kvasir-SEG/images', str(kvasir_dir / 'images'))
    shutil.copytree('/tmp/kvasir_raw/Kvasir-SEG/masks',  str(kvasir_dir / 'masks'))
    print(f'Kvasir-SEG placed at {kvasir_dir}')
else:
    print(f'Kvasir-SEG already at {kvasir_dir}')

imgs = list((kvasir_dir / 'images').glob('*'))
msks = list((kvasir_dir / 'masks').glob('*'))
print(f'  Images: {len(imgs)}  |  Masks: {len(msks)}')

In [ ]:
# -------------------------------------------------------------------------
# Option B: PraNet Google Drive package (all 5 datasets, correct structure)
# -------------------------------------------------------------------------
# 1. Go to https://github.com/DengPingFan/PraNet and find the Google Drive link
#    in the README under "Data preparation".
# 2. Paste the file ID into GDRIVE_FILE_ID below.
# 3. Run this cell — gdown will download and unzip automatically.

GDRIVE_FILE_ID = ''   # <-- paste file ID here

if GDRIVE_FILE_ID:
    import gdown
    zip_path = '/tmp/polyp_data.zip'
    if not Path(zip_path).exists():
        gdown.download(id=GDRIVE_FILE_ID, output=zip_path, quiet=False)
    !unzip -q {zip_path} -d {DATA_ROOT}
    print('PraNet package extracted.')
else:
    print('GDRIVE_FILE_ID not set — using Kvasir-SEG only for now.')
    print('Set it to get CVC-ClinicDB, CVC-ColonDB, ETIS-Larib, CVC-300.')

## 3. Verify Split Sizes

PraNet protocol: **train on 900 Kvasir + 550 CVC-ClinicDB (1,450 total)**,
test on held-out seen splits and three fully unseen datasets.

In [ ]:
from src.data import build_splits
from src.data.dataset import verify_splits

try:
    splits = build_splits(DATA_ROOT, seed=42)
    verify_splits(splits)
except FileNotFoundError as e:
    print(f'\nSome datasets not yet downloaded: {e}')
    print('Run Option B above (PraNet Google Drive package) to get all five datasets.')
    # Build partial splits from what we have
    splits = None

In [ ]:
# Quick sanity check: confirm no overlap between train and any test split
if splits is not None:
    train_names = {p.name for p in splits['train']['image_paths']}
    for split_name in ['seen_kvasir', 'seen_clinicdb', 'cvc_colondb', 'etis_larib', 'cvc_300']:
        if split_name not in splits:
            continue
        test_names = {p.name for p in splits[split_name]['image_paths']}
        overlap = train_names & test_names
        status = '✓ No overlap' if not overlap else f'✗ OVERLAP: {len(overlap)} images!'
        print(f'{split_name:<20}: {status}')

## 4. Visualise Samples

In [ ]:
def show_samples(image_paths, mask_paths, title, n=4):
    n = min(n, len(image_paths))
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i in range(n):
        img = np.array(Image.open(image_paths[i]).convert('RGB'))
        msk = np.array(Image.open(mask_paths[i]).convert('L'))
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Image {i+1}', fontsize=9)
        axes[0, i].axis('off')
        axes[1, i].imshow(msk, cmap='gray')
        axes[1, i].set_title(f'Mask {i+1}', fontsize=9)
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

# Show Kvasir-SEG samples (always available after Option A)
kvasir_imgs = sorted((kvasir_dir / 'images').glob('*'))
kvasir_msks = sorted((kvasir_dir / 'masks').glob('*'))
show_samples(kvasir_imgs, kvasir_msks, 'Kvasir-SEG (train dataset)', n=4)

In [ ]:
# Show samples from each available dataset
dataset_dirs = {
    'CVC-ClinicDB (train)': DATA_ROOT / 'TrainDataset' / 'CVC-ClinicDB',
    'CVC-ColonDB (unseen)': DATA_ROOT / 'TestDataset' / 'CVC-ColonDB',
    'ETIS-Larib (unseen)':  DATA_ROOT / 'TestDataset' / 'ETIS-Larib',
    'CVC-300 (unseen)':     DATA_ROOT / 'TestDataset' / 'CVC-300',
}

for name, folder in dataset_dirs.items():
    if not (folder / 'images').exists():
        print(f'  {name}: not yet downloaded')
        continue
    imgs = sorted((folder / 'images').glob('*'))
    msks = sorted((folder / 'masks').glob('*'))
    show_samples(imgs, msks, name, n=4)

## 5. Augmentation Preview

In [ ]:
from src.data import get_train_transform

transform = get_train_transform(img_size=352)

# Pick one image and show 6 augmented versions
raw_img = np.array(Image.open(kvasir_imgs[0]).convert('RGB'))
raw_msk = np.array(Image.open(kvasir_msks[0]).convert('L'))
raw_msk_f = (raw_msk > 127).astype(np.float32)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Augmentation preview (same image, 6 random crops)', fontsize=13)

# column 0: original
axes[0, 0].imshow(raw_img)
axes[0, 0].set_title('Original')
axes[1, 0].imshow(raw_msk, cmap='gray')
axes[1, 0].set_title('GT mask')
for ax in [axes[0,0], axes[1,0]]:
    ax.axis('off')

for i in range(1, 6):
    out = transform(image=raw_img, mask=raw_msk_f)
    # un-normalise for display
    img_t = out['image'].permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_display = np.clip(img_t * std + mean, 0, 1)
    axes[0, i].imshow(img_display)
    axes[0, i].set_title(f'Aug {i}')
    axes[1, i].imshow(out['mask'].numpy(), cmap='gray')
    axes[1, i].set_title(f'Mask {i}')
    for ax in [axes[0,i], axes[1,i]]:
        ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Evaluation Metrics — Smoke Test

All six metrics used in the PraNet evaluation protocol, tested on:
- A **perfect prediction** (pred = gt)
- A **random prediction** (pred ~ Uniform[0,1])
- An **all-zeros prediction** (model predicts nothing)

In [ ]:
from src.metrics import dice_score, iou_score, mae_score, weighted_f_measure, s_measure, e_measure

# Load a real GT mask for testing
gt = (np.array(Image.open(kvasir_msks[0]).convert('L')) > 127).astype(np.float32)
rng = np.random.default_rng(0)

cases = {
    'Perfect  (pred=gt)': gt,
    'Random   (uniform)': rng.uniform(0, 1, gt.shape).astype(np.float32),
    'All zeros          ': np.zeros_like(gt),
}

print(f'{"Case":<22} {"Dice":>7} {"IoU":>7} {"MAE":>7} {"wFm":>7} {"Sm":>7} {"Em":>7}')
print('-' * 70)
for name, pred in cases.items():
    pred_bin = (pred >= 0.5).astype(np.float32)
    print(
        f'{name:<22}'
        f' {dice_score(pred_bin, gt):>7.4f}'
        f' {iou_score(pred_bin, gt):>7.4f}'
        f' {mae_score(pred, gt):>7.4f}'
        f' {weighted_f_measure(pred_bin, gt):>7.4f}'
        f' {s_measure(pred, gt):>7.4f}'
        f' {e_measure(pred, gt):>7.4f}'
    )

In [ ]:
# MetricTracker accumulates per-image scores across a split
from src.metrics import MetricTracker

tracker = MetricTracker()

# Simulate scoring 10 images with random predictions
for i in range(10):
    gt_i = (np.array(Image.open(kvasir_msks[i]).convert('L')) > 127).astype(np.float32)
    pred_i = rng.uniform(0, 1, gt_i.shape).astype(np.float32)
    tracker.update(pred_i, gt_i)

print('MetricTracker summary over 10 random predictions:')
print(' ', tracker.summary())

## 7. Dataset Statistics

In [ ]:
# Polyp size distribution in Kvasir-SEG
sizes = []
for msk_path in kvasir_msks:
    m = np.array(Image.open(msk_path).convert('L'))
    polyp_px = (m > 127).sum()
    total_px = m.size
    sizes.append(100 * polyp_px / total_px)

plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=30, edgecolor='black', color='steelblue', alpha=0.8)
plt.xlabel('Polyp area (% of image)')
plt.ylabel('Count')
plt.title('Kvasir-SEG — polyp size distribution (n=1000)')
plt.axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean={np.mean(sizes):.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean polyp area: {np.mean(sizes):.1f}%  |  Median: {np.median(sizes):.1f}%  |  '
      f'Std: {np.std(sizes):.1f}%')

## Phase 1 Summary

| Deliverable | Status |
|---|---|
| Repository & dependency setup | ✅ |
| Kvasir-SEG downloaded & verified | ✅ |
| PraNet data package (all 5 datasets) | ⬜ Set `GDRIVE_FILE_ID` above |
| Train/seen/unseen split builder | ✅ (`src/data/dataset.py`) |
| No train/test overlap verified | ✅ |
| All 6 evaluation metrics | ✅ (`src/metrics/segmentation.py`) |
| Augmentation pipeline | ✅ (`src/data/transforms.py`) |

**Next:** `02_unet_baseline.ipynb` — train the specialist baseline.